# Week 3 recap — Sheet 02 SOLUTIONS: the rows that quietly disappear

Executed in the lab image. Every quoted number is what it actually printed.

Question 3 is the one to internalise: an aggregate that silently reports fewer
rows than it was given, with no warning and no error.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Week 3 recap, sheet 02 — Rows that disappear. Run this once.
import glob
import numpy as np
import pandas as pd

BRONZE = "data/bronze/"
orders = (pd.concat([pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))],
                    ignore_index=True)
            .drop_duplicates(subset="LineID"))
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")

# The real data has no missing keys. A real CRM export does, so this recap
# introduces them deliberately -- 60 customers lose their segment, and 25
# order lines lose their product. Both are DETERMINISTIC (every 30th and
# every 320th row), so every number below is reproducible.
customers_gappy = customers.copy()
customers_gappy.loc[customers_gappy.index % 30 == 0, "CustomerSegment"] = np.nan

orders_gappy = orders.reset_index(drop=True).copy()
orders_gappy.loc[orders_gappy.index % 320 == 0, "ProductID"] = np.nan

print("orders          ", orders.shape)
print("customers_gappy ", customers_gappy.shape,
      "| null CustomerSegment:", int(customers_gappy.CustomerSegment.isna().sum()))
print("orders_gappy    ", orders_gappy.shape,
      "| null ProductID:", int(orders_gappy.ProductID.isna().sum()))

PART A — the aggregate that shortens

### Question 1

Join `orders` to `customers_gappy` with a left join, then count order lines per `CustomerSegment` with a plain `groupby`. Print the result, its total, and the row count that went in.
> **NOTE:** add the reported numbers up before reading on.

In [ ]:
j = orders.merge(customers_gappy, on="CustomerID", how="left",
                 validate="many_to_one")
g = j.groupby("CustomerSegment").size()
print(g.rename("lines").to_string())
print()
print("rows that went in:   ", len(j))
print("rows reported:       ", int(g.sum()))
print("missing from the report:", len(j) - int(g.sum()))

```
CustomerSegment
Consumer          1513
Corporate         2876
Home Office       1923
Small Business    1525

rows that went in:    8060
rows reported:        7837
missing from the report: 223
```

Four segments, sensible counts, sorted, formatted. Nothing about it looks wrong.

Add them up: 1513 + 2876 + 1923 + 1525 = **7,837**. The join produced **8,060**
rows.

**223 order lines are absent from a report that raised nothing.**

This is day 4 worksheet 01 exactly — there it was 316 of 2,400 enrollments — and
it is the single most common silent failure in pandas. `groupby` defaults to
`dropna=True`, so any row whose group key is null is discarded before counting.

The reason it survives review is that the output has every property of a
finished answer. There is no null row to notice, no warning, no exception. The
only symptom is a total nobody computed.

**Print the row count next to every aggregate you produce.** One line, and it is
the difference between this being visible and not.

### Question 2

Find them. Count the joined rows whose `CustomerSegment` is null, and confirm the arithmetic.

In [ ]:
j = orders.merge(customers_gappy, on="CustomerID", how="left",
                 validate="many_to_one")
missing = int(j.CustomerSegment.isna().sum())
print("joined rows:            ", len(j))
print("null CustomerSegment:   ", missing)
print("reported by groupby:    ", int(j.groupby("CustomerSegment").size().sum()))
print("%d + %d = %d" % (int(j.groupby("CustomerSegment").size().sum()),
                        missing, len(j)))
print()
print("share of the business missing from the report: %.1f%%"
      % (100 * missing / len(j)))

```
joined rows:             8060
null CustomerSegment:    223
reported by groupby:     7837
7837 + 223 = 8060

share of the business missing from the report: 2.8%
```

The arithmetic closes exactly, which is what makes the diagnosis certain rather
than probable.

**2.8% of order lines** vanished, from 62 customers with no segment recorded. The
ratio is worth noticing: 62 customers out of 1,832 is 3.4%, and they account for
2.8% of lines — so the missing customers are ordinary ones, not a special case.
Had it been 62 customers accounting for 40% of lines, the report would be
useless rather than merely wrong.

And note where the null came from: **not** from `orders`. Every order line has a
`CustomerID`, and the join matched every one. The null is an *attribute* of the
matched customer — the CRM export simply has gaps.

That distinction matters for the fix. This is not a broken join (nothing to
repair upstream); it is a dimension with missing values, and the answer is to
give them a label rather than to hunt for the row. Question 3.

### Question 3

Fix it two ways: `groupby(..., dropna=False)`, and `fillna("Unknown")` before grouping. Print both, and say which you would ship.
> **NOTE:** day 4 worksheet 01 made this exact choice, and day 4 worksheet 08 made it a loading rule.

In [ ]:
j = orders.merge(customers_gappy, on="CustomerID", how="left",
                 validate="many_to_one")
print("A) dropna=False:")
a = j.groupby("CustomerSegment", dropna=False).size()
print(a.rename("lines").to_string())
print("   total:", int(a.sum()))
print()
print("B) fillna('Unknown') first:")
b = j.assign(CustomerSegment=j.CustomerSegment.fillna("Unknown")) \
     .groupby("CustomerSegment").size()
print(b.rename("lines").to_string())
print("   total:", int(b.sum()))
print()
print("both reconcile to", len(j), "-- but only one survives a CSV export,")
print("a BI tool, or a join on that column.")

```
A) dropna=False:            B) fillna('Unknown') first:
Consumer          1513      Consumer          1513
Corporate         2876      Corporate         2876
Home Office       1923      Home Office       1923
Small Business    1525      Small Business    1525
NaN                223      Unknown            223
   total: 8060                 total: 8060
```

Both reconcile to 8,060. Both make the missing rows visible. They are **not**
equivalent.

**`dropna=False` fixes the symptom, in one place.** The next `groupby` someone
writes against this data has the bug again, because the default is unchanged.
And `NaN` as a category label is awkward downstream: it exports to an empty CSV
cell, sorts unpredictably, displays as blank in most BI tools, and **will not
join** — `NaN != NaN`, so a later merge on that column drops it again.

**`fillna("Unknown")` fixes the data, once.** `Unknown` is a value. It groups,
joins, exports, sorts and displays like any other category, and it appears in
every report built on the table without anyone opting in.

Ship B — and better, ship it at **load time**, not in the query. That is what day
4 worksheet 08 did with `dim_program.program_category` and what day 4 worksheet
06 recorded as rule R4. Once the dimension says `Unknown`, all 223 rows are
countable forever and somebody eventually asks why 2.8% of lines have no segment
— which is the question that gets the CRM fixed.

Use `dropna=False` when you are *investigating*. Use `fillna` when you are
*building*.

PART B — the join that filters

### Question 4

Join `orders_gappy` to `products` three ways — `inner`, `left`, and `left` with `indicator=True` — and print the row count of each plus the indicator breakdown.

In [ ]:
for how in ("inner", "left"):
    n = len(orders_gappy.merge(products, on="ProductID", how=how))
    print("  how=%-6s %5d rows" % (how, n))
print()
j = orders_gappy.merge(products, on="ProductID", how="left", indicator=True)
print(j._merge.value_counts().to_string())
print()
print("orders_gappy rows: ", len(orders_gappy))
print("lost by inner join:",
      len(orders_gappy) - len(orders_gappy.merge(products, on="ProductID", how="inner")))

```
  how=inner   8034 rows
  how=left    8060 rows

_merge
both          8034
left_only       26
right_only       0

orders_gappy rows:  8060
lost by inner join: 26
```

**26 rows, and the inner join simply deletes them.**

These are order lines whose `ProductID` is null, so there is nothing to match.
`left` keeps them with null product columns — findable. `inner` removes them —
gone, with the row count 26 lower and no indication why.

That is day 4 worksheet 07's `AssertionError: fact table has 2219 rows`, at
smaller scale: an inner join that produced a table passing every *internal*
check. Unique grain, no nulls — of course no nulls, the rows that would have had
them were deleted.

`indicator=True` is the diagnostic and costs nothing: **8,034 `both`, 26
`left_only`, 0 `right_only`.** Three numbers that say precisely what the join
did.

The three ways this reaches production are all ordinary:

- `how="inner"` typed from habit
- `how=` omitted — **pandas defaults to `inner`**
- a `WHERE` on a joined column, which turns a left join into an inner one by
  discarding the nulls it just created

None of the three looks like a bug in review, which is why the check has to be
mechanical rather than visual.

### Question 5

Put money on it. Compute `SUM(Sales)` over the inner join and over `orders_gappy` itself, and print the difference and the percentage.
> **NOTE:** a revenue total that is too *small* is the hardest kind of wrong to notice.

In [ ]:
inner = orders_gappy.merge(products, on="ProductID", how="inner")
true = orders_gappy.Sales.sum()
print("SUM(Sales), all orders:      %13.2f" % true)
print("SUM(Sales), after inner join:%13.2f" % inner.Sales.sum())
print("lost:                        %13.2f  (%.3f%%)"
      % (true - inner.Sales.sum(), 100 * (true - inner.Sales.sum()) / true))
print()
print("rows: %d -> %d" % (len(orders_gappy), len(inner)))

```
SUM(Sales), all orders:        13570810.63
SUM(Sales), after inner join:  13524709.11
lost:                             46101.51  (0.340%)

rows: 8060 -> 8034
```

**46,101.51 of revenue, gone, from 26 rows — 0.340%.**

That magnitude is the danger, and it is the same one that keeps recurring: day 1
worksheet 17's re-delivery was 0.462%, day 3's duplicate transactions were 0.70%,
day 4's join fan-out was 0.414%. All under one percent. All invisible.

But a *shortfall* is harder to catch than an *overstatement*, and the reason is
psychological rather than technical. A revenue number that is too high invites
scrutiny — someone asks whether it is real. A number that is too low reads as a
soft quarter. Nobody investigates a total for being disappointing.

There is also no internal signal. The joined table is entirely
self-consistent: 8,034 rows, every one valid, every product matched. The only way
to know is to compare against something computed **outside** it — which is
question 9.

The rule: **after any join, compare the row count and a control total against the
inputs.** Two lines. If either moved and you did not intend it, stop.

### Question 6

Do it properly: left join, then route the unmatched rows to an `Unknown` product rather than losing them. Print the row count, the total, and how many landed on `Unknown`.
> **NOTE:** day 4 worksheet 08's `Unknown` dimension member, applied here.

In [ ]:
dim = products[["ProductID", "ProductCategory"]].copy()
dim = pd.concat([pd.DataFrame([{"ProductID": -1, "ProductCategory": "Unknown"}]),
                 dim], ignore_index=True)

o = orders_gappy.copy()
o["ProductID"] = o.ProductID.fillna(-1)
j = o.merge(dim, on="ProductID", how="left", validate="many_to_one")

print("rows:                  %5d (orders_gappy had %d)" % (len(j), len(orders_gappy)))
print("SUM(Sales):     %13.2f (true %13.2f)" % (j.Sales.sum(), orders_gappy.Sales.sum()))
print("null ProductCategory:  %5d" % int(j.ProductCategory.isna().sum()))
print()
print("rows on the Unknown member:", int((j.ProductCategory == "Unknown").sum()))
print()
print(j.groupby("ProductCategory").size().rename("lines").to_string())

```
rows:                   8060 (orders_gappy had 8060)
SUM(Sales):       13570810.63 (true   13570810.63)
null ProductCategory:      0

rows on the Unknown member: 26

ProductCategory
Furniture          1571
Office Supplies    4566
Technology         1897
Unknown              26
```

**Every row kept, the total exact, and zero nulls** — with the 26 problem rows
sitting in plain sight under `Unknown`.

Three things had to line up:

**An `Unknown` member in the dimension**, keyed `-1`. Day 4 worksheet 08's
convention: a negative key cannot collide with a generated sequence and is
instantly recognisable in a result.

**`fillna(-1)` on the fact's key before the join**, so the join itself resolves
to that member. Cleaner than joining, getting a null, and patching afterwards —
the intent is visible in the code.

**`how="left"` and `validate="many_to_one"`**, so the row count cannot change and
a duplicated dimension row would raise rather than inflate.

Compare the three approaches on the same 26 rows:

| approach | rows | total | anomaly visible? |
|---|---|---|---|
| inner join | 8,034 | 13,524,709.11 | **no** — deleted |
| left join, null key | 8,060 | 13,570,810.63 | only if someone checks |
| left join, `Unknown` | 8,060 | 13,570,810.63 | **yes** — 26 rows, labelled |

The middle one is worse than it looks: the rows are in the table and absent from
every report that groups by category, because `groupby` drops them (question 1).
So it fixes the total and not the report.

`Unknown: 26` is a number somebody will eventually ask about. That is the whole
point.

PART C — values that are not what they look like

### Question 7

Day 3 found values carrying quote characters, so `WHERE PRIORITY = 'High'` matched nothing. Reproduce the shape of that here: pad `ProductCategory` with a trailing space on every row, then filter for `== "Technology"` and count.
> **NOTE:** an empty result is not an error. It reads exactly like "there were none".

In [ ]:
j = orders.merge(products[["ProductID", "ProductCategory"]], on="ProductID",
                 how="left", validate="many_to_one")
padded = j.assign(ProductCategory=j.ProductCategory + " ")

print("rows matching 'Technology'          :", int((padded.ProductCategory == "Technology").sum()))
print("rows matching 'Technology ' (padded):", int((padded.ProductCategory == "Technology ").sum()))
print()
print("distinct values, repr()'d:")
for v in sorted(padded.ProductCategory.unique()):
    print("   ", repr(v))
print()
print("after .str.strip():",
      int((padded.ProductCategory.str.strip() == "Technology").sum()))

```
rows matching 'Technology'          : 0
rows matching 'Technology ' (padded): 1902

distinct values, repr()'d:
    'Furniture '
    'Office Supplies '
    'Technology '

after .str.strip(): 1902
```

**Zero rows** — and 1,902 of them are sitting right there.

This is day 3 worksheet 03 in a different costume. There the values arrived from
a CSV wrapped in quote characters, so `WHERE PRIORITY = 'High'` returned nothing
while 21,117 matching rows sat in the table. Here it is a trailing space. The
mechanism and the consequence are identical.

What makes it dangerous is that **an empty result is not an error**. `0` is a
valid count. It reads exactly like "there were no Technology orders", which is a
perfectly plausible sentence about a business, and a filter returning nothing
gives you no reason to doubt it.

`repr()` is what breaks the illusion. Printed normally, `Technology ` and
`Technology` are indistinguishable — the space is invisible by definition.
`repr()` puts quotes around it and the difference is obvious at a glance.

**So: any time a filter returns zero rows and you expected some, `repr()` the
distinct values before believing it.** The usual culprits are trailing
whitespace, embedded quote characters, non-breaking spaces, and case.

And fix it at load, not in the query — `.str.strip()` once beats
`WHERE TRIM(col) = ...` in every query forever.

### Question 8

Write the check that would have caught question 7 in seconds: for every low-cardinality text column in the joined table, print its distinct values with `repr()`.
> **NOTE:** cheap enough to run on arrival, every time. It is how you find quoting, padding and casing before a query does.

In [ ]:
j = orders.merge(customers, on="CustomerID", how="left") \
          .merge(products[["ProductID", "ProductCategory"]], on="ProductID", how="left")

for col in j.columns:
    if j[col].dtype == "object" or str(j[col].dtype) == "str":
        n = j[col].nunique()
        if n <= 12:
            print("%-18s %2d distinct" % (col, n))
            for v in sorted(j[col].dropna().unique()):
                print("      %r" % v)

```
OrderPriority       5 distinct      Region              8 distinct
      'Critical'                          'Atlantic'
      'High'                              'Northwest Territories'
      'Low'                               'Nunavut'
      'Medium'                            'Ontario'
      'Not Specified'                     'Prarie'
                                          'Quebec'
ShipMode            3 distinct            'West'
      'Delivery Truck'                    'Yukon'
      'Express Air'
      'Regular Air'                 ProductCategory     3 distinct
                                          'Furniture'
CustomerSegment     4 distinct            'Office Supplies'
      'Consumer'                          'Technology'
      'Corporate'
      'Home Office'
      'Small Business'
```

Five columns, 23 values, and the check took seconds. Everything is clean —
no padding, no quotes, no casing variants.

Except one. Look at the Region list: **`'Prarie'`**.

That is a misspelling of *Prairie*, and it is **real** — it is in the source
superstore data, not something this sheet introduced. Nobody planted it and
nobody had noticed it until this check ran.

It is the good kind of finding, because it is currently harmless and will not
stay that way. Today it is one consistent spelling, so every report groups it
correctly under a slightly wrong label. The moment anyone loads a second source
that spells it *Prairie*, the region silently splits into two and every regional
total is wrong — and that failure will look like a data problem in the new
source rather than in this one.

**This is the cheapest check in data engineering.** Distinct values of every
low-cardinality text column, `repr()`'d, on arrival. It finds quoting, padding,
casing, misspellings and unexpected new categories, and it fits on one screen.

Run it on every new dataset before you write a single query against it.

PART D — reconcile against something outside

### Question 9

Build a small pipeline — join, filter to 2012, aggregate by category — and print a reconciliation table: row count and `SUM(Sales)` at each step, with the difference explained at every stage.

In [ ]:
j = orders.merge(products[["ProductID", "ProductCategory"]], on="ProductID",
                 how="left", validate="many_to_one")
y2012 = j[pd.to_datetime(j.OrderDate).dt.year == 2012]
gold = y2012.groupby("ProductCategory").agg(lines=("LineID", "size"),
                                            sales=("Sales", "sum")).reset_index()

print("%-22s %8s %15s" % ("STEP", "ROWS", "SUM(Sales)"))
print("%-22s %8d %15.2f" % ("orders", len(orders), orders.Sales.sum()))
print("%-22s %8d %15.2f" % ("+ products (left)", len(j), j.Sales.sum()))
print("%-22s %8d %15.2f" % ("filter year == 2012", len(y2012), y2012.Sales.sum()))
print("%-22s %8d %15.2f" % ("groupby category", len(gold), gold.sales.sum()))
print()
print("join changed the total:", round(j.Sales.sum(), 2) != round(orders.Sales.sum(), 2))
print("groupby changed the 2012 total:",
      round(gold.sales.sum(), 2) != round(y2012.Sales.sum(), 2))
print("the filter is the only step that should drop rows, and it dropped",
      len(j) - len(y2012))

```
STEP                       ROWS      SUM(Sales)
orders                     8060     13570810.63
+ products (left)          8060     13570810.63
filter year == 2012        2020      3356203.19
groupby category              3      3356203.19

join changed the total: False
groupby changed the 2012 total: False
the filter is the only step that should drop rows, and it dropped 6040
```

Four steps, and each one is accounted for.

**The join changed nothing** — 8,060 rows in, 8,060 out, total identical. That is
what a correct many-to-one join looks like, and printing it is how you know.

**The filter dropped 6,040 rows** and reduced the total, which is exactly its
job. A filter is the *only* step in this pipeline permitted to lose rows, and
that is the property worth stating out loud: if any other step changes the count,
it is a bug.

**The `groupby` preserved the total** — 3,356,203.19 before and after. 2,020 rows
became 3, and not a cent moved. That is the check questions 1 and 10 fail: a
`groupby` that reconciles is one where no key was null.

This is the reconciliation habit from day 1 worksheet 17 (bronze → silver →
gold) and day 3 worksheet 05, in miniature. The value is not any single number;
it is that **every difference has a named cause**. An unexplained gap is an
incident. The same gap with "the filter, 6,040 rows" beside it is a documented
property of the pipeline.

Store this table per run. It turns "did the load work?" into a glance, and "when
did this number start being wrong?" into a query.

### Question 10

Finally, assert that the report in question 1 accounts for every row it was given. **This is supposed to fail.** Read the number and say what a reader of that report would have concluded.

In [ ]:
j = orders.merge(customers_gappy, on="CustomerID", how="left",
                 validate="many_to_one")
reported = int(j.groupby("CustomerSegment").size().sum())

print("rows given to the report:", len(j))
print("rows the report accounts for:", reported)
print("silently absent:", len(j) - reported)
print()
print("SUM(Sales) reported:  %13.2f"
      % j.dropna(subset=["CustomerSegment"]).Sales.sum())
print("SUM(Sales) true:      %13.2f" % j.Sales.sum())
print()
assert reported == len(j), (
    "the segment report accounts for %d of %d rows -- %d disappeared into a "
    "null key" % (reported, len(j), len(j) - reported))

```
rows given to the report: 8060
rows the report accounts for: 7837
silently absent: 223

SUM(Sales) reported:    13206299.53
SUM(Sales) true:        13570810.63

AssertionError: the segment report accounts for 7837 of 8060 rows -- 223
disappeared into a null key
```

The assertion is the only thing on this sheet that objected.

Look at what a reader of that report would have concluded. Four segments, plausible
counts, and revenue of **13,206,299.53** — a specific, confident, wrong number.
The true figure is **13,570,810.63**, so the report understates revenue by the
difference between those two figures while looking complete.

Nothing in the output hints at it. There is no null row, no warning, no
exception. The reader has no reason to add the counts up, and adding them up is
the only way to find out.

**The check is one line**, and it belongs next to every aggregate:

```python
assert grouped.sum() == len(source), "rows disappeared into a null key"
```

That is the recap of the whole sheet: an aggregate must account for every row it
was given, and a join must not change the row count unless you asked it to.

**What disappeared, and how:**

| | rows | cost |
|---|---|---|
| `groupby` on a null key | **223** (2.8%) | reported revenue 13,206,299.53 against a true 13,570,810.63 |
| inner join on a null key | **26** | **46,101.51 (0.340%)** deleted from the total |
| a trailing space | **1,902** | a filter returning `0` that reads as "none found" |
| `'Prarie'` | 0 today | a real misspelling, waiting for a second source to split a region |

**Four defences, in the order they pay off:**

1. **Print the row count beside every aggregate.** Catches the `groupby`.
2. **`fillna("Unknown")` at load, not `dropna=False` in the query.** Fixes it once.
3. **`how="left"` plus an `Unknown` member.** Nothing is deleted by a join.
4. **`repr()` the distinct values of every text column on arrival.** Catches
   padding, quoting, casing and misspellings before a query does.

Sheet 03 takes the last piece of the week: numbers that survive all of the above
and are still wrong, because they were averaged when they should not have been.